In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import UNet2DModel, AutoencoderKL
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T
from pathlib import Path
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Dataset

In [2]:
DATASET = "mnist"

# --------------------------------------------------
# Transforms
# --------------------------------------------------

if DATASET == "mnist":

    tfm = T.Compose([
        T.Resize((32, 32)),
        T.Grayscale(num_output_channels=3),
        T.ToTensor(),
        T.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ])

elif DATASET in ["cifar10", "cifar100"]:

    tfm = T.Compose([
        T.ToTensor(),
        T.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ])

elif DATASET == "stl10":

    tfm = T.Compose([
        T.Resize((32, 32)),
        T.ToTensor(),
        T.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ])


# --------------------------------------------------
# Dataset
# --------------------------------------------------

if DATASET == "cifar100":

    dataset = CIFAR100(
        root="datasets/cifar100/",
        train=True,
        download=True,
        transform=tfm,
    )

    test_set = CIFAR100(
        root="datasets/cifar100/",
        train=False,
        download=True,
        transform=tfm,
    )

elif DATASET == "cifar10":

    dataset = CIFAR10(
        root="datasets/cifar10/",
        train=True,
        download=True,
        transform=tfm,
    )

    test_set = CIFAR10(
        root="datasets/cifar10/",
        train=False,
        download=True,
        transform=tfm,
    )

elif DATASET == "stl10":

    dataset = STL10(
        root="datasets/stl10/",
        split="train",
        download=True,
        transform=tfm,
    )

    test_set = STL10(
        root="datasets/stl10/",
        split="test",
        download=True,
        transform=tfm,
    )

elif DATASET == "mnist":

    dataset = MNIST(
        root="datasets/mnist/",
        train=True,
        download=True,
        transform=tfm,
    )

    test_set = MNIST(
        root="datasets/mnist/",
        train=False,
        download=True,
        transform=tfm,
    )

### Utils

In [3]:

# --------------------------------------------------
# Continuous VP diffusion
#
# dx = -1/2 beta(t) x dt + sqrt(beta(t)) dW
#
# beta(t) = beta_min + t (beta_max - beta_min)
# --------------------------------------------------

beta_min = 0.1
beta_max = 20.0


def vp_alpha_sigma(t):
    """
    t: shape (B,), with t in [0, 1]

    Returns:
        alpha(t), sigma(t), each shape (B,)
    """

    log_alpha = (
        -0.5 * beta_min * t
        -0.25 * (beta_max - beta_min) * t**2
    )

    alpha = torch.exp(log_alpha)
    sigma = torch.sqrt(1.0 - alpha**2)

    return alpha, sigma


def add_vp_noise(x0, t):
    """
    x_t = alpha(t) x_0 + sigma(t) eps
    """

    eps = torch.randn_like(x0)

    alpha, sigma = vp_alpha_sigma(t)

    alpha = alpha[:, None, None, None]
    sigma = sigma[:, None, None, None]

    xt = alpha * x0 + sigma * eps

    return xt, eps




### Model

In [5]:
ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
ae_model = ae_model.to(device).eval()
ae_model.requires_grad_(False)
ae_scaling = ae_model.config.scaling_factor

/home/schimmenti/miniconda3/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [20]:
with torch.no_grad():
    for batch in DataLoader(test_set, batch_size=1, shuffle=False):
        x0, _ = batch
        x0 = x0.to(device)
        latents = ae_model.encode(x0).latent_dist.sample()
        C_latents, H_latents, W_latents = latents.shape[1:]
        latents_shape = (C_latents, H_latents, W_latents)
        break

In [ ]:
class MLPBlock(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        activation=nn.SiLU,
        norm=nn.RMSNorm,
        dropout=0.0,
    ):
        super().__init__()

        layers = [
            nn.Linear(d_in, d_out),
        ]

        if norm is not None:
            layers.append(norm(d_out))

        if activation is not None:
            layers.append(activation())

        if dropout > 0:
            layers.append(nn.Dropout(dropout))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
class SwiGLUBlock(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        norm=nn.RMSNorm,
        dropout=0.0,
    ):
        super().__init__()

        self.norm = norm(d_in) if norm is not None else nn.Identity()
        self.proj = nn.Linear(d_in, 2 * d_out)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        x = self.norm(x)
        gate, value = self.proj(x).chunk(2, dim=-1)
        return self.dropout(F.silu(gate) * value)
class GaussianLayer(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dims=(128, 128),
        block=MLPBlock,
        logvar_min=-8.0,
        logvar_max=4.0,
    ):
        super().__init__()

        dims = [input_dim, *hidden_dims]

        self.net = nn.Sequential(*[
            block(d_in, d_out)
            for d_in, d_out in zip(dims[:-1], dims[1:])
        ])

        self.mu = nn.Linear(dims[-1], output_dim)
        self.logvar = nn.Linear(dims[-1], output_dim)

        self.logvar_min = logvar_min
        self.logvar_max = logvar_max

    def forward(self, x):
        h = self.net(x)

        mu = self.mu(h)

        raw_logvar = self.logvar(h)
        logvar = self.logvar_min + (
            self.logvar_max - self.logvar_min
        ) * torch.sigmoid(raw_logvar)

        return mu, logvar

class AttentionWeights(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.memory = nn.Linear(input_dim, hidden_dim)
        nn.init.normal_(self.memory.weight, mean=0.0, std=input_dim**-0.5)
        nn.init.zeros_(self.memory.bias)

    def forward(self, x):
        """
        x: shape (B, input_dim)
        """
        x = self.memory(x)

        # Compute attention scores
        attn_scores = torch.matmul(x, self.memory.t())  # shape (B, hidden_dim)

        # Apply softmax to get attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)  # shape (B, hidden_dim)

        return attn_weights     

In [ ]:
decoder = GaussianLayer(
    input_dim=C_latents * H_latents * W_latents,
    output_dim=32,
    hidden_dims=(256, 256),
    block=SwiGLUBlock,
    logvar_min=-8.0,
    logvar_max=4.0,
)
memory_bank = AttentionWeights(
    input_dim=C_latents * H_latents * W_latents,
    hidden_dim=128,
)